In [1]:
import pandas as pd
import numpy as np

In [3]:
clean_data = pd.read_parquet("../cleaned_data/clean_yellow_tripdata_2026-01.parquet")

In [4]:
df = clean_data.copy()

In [5]:
df.shape

(1853629, 35)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1853629 entries, 0 to 1853628
Data columns (total 34 columns):
 #   Column                        Dtype          
---  ------                        -----          
 0   vendor_id                     int8           
 1   pickup_datetime               datetime64[us] 
 2   dropoff_datetime              datetime64[us] 
 3   passenger_count               int8           
 4   trip_distance                 float32        
 5   fare_type_id                  float64        
 6   store_and_fwd_flag            str            
 7   pickup_location_id            int32          
 8   dropoff_location_id           int32          
 9   payment_type                  int64          
 10  fare_amount                   float64        
 11  extra_amount                  float64        
 12  mta_tax_amount                float64        
 13  tip_amount                    float64        
 14  tolls_amount                  float64        
 15  improvement_surcharge_amou

In [9]:
df['fare_type_id'].isna().sum()

np.int64(0)

In [8]:
df['fare_type_id'].value_counts(dropna=False)

fare_type_id
1.0     1639242
99.0     105594
2.0       76851
5.0       14485
3.0        9451
4.0        8005
6.0           1
Name: count, dtype: int64

In [11]:
df[df['fare_type_id'] == 6].T

,1511470
vendor_id,1
pickup_datetime,2026-01-25 02:05:19
dropoff_datetime,2026-01-25 02:09:53
passenger_count,1
trip_distance,1.1
fare_type_id,6.0
store_and_fwd_flag,N
pickup_location_id,160
dropoff_location_id,160
payment_type,3


In [22]:
pd.crosstab(df["fare_type_id"], df["passenger_count"])

passenger_count,1,2,3,4,5,6
fare_type_id,,,,,,
1,1340958,212966,46097,29309,6571,3341
2,49043,21785,3246,2366,219,192
3,6232,1735,1121,339,15,9
4,6062,1165,400,351,17,10
5,8507,2787,782,2402,5,2
6,1,0,0,0,0,0
99,105594,0,0,0,0,0


# change data type

In [14]:
df['fare_type_id'] = df['fare_type_id'].astype('int8')

In [20]:
df.memory_usage().sum()/(1024*1024)

np.float64(501.51911067962646)

# cross check fare type with loactions

In [49]:
df[
    (df['fare_type_id'] == 2) &
    ~(
        df['pickup_zone'].str.contains('JFK', case=False) |
        df['dropoff_zone'].str.contains('JFK', case=False)
    )
].sample()


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,fare_type_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,...,pickup_borough,pickup_longitude,pickup_latitude,dropoff_zone,dropoff_borough,dropoff_longitude,dropoff_latitude,great_circle_distance,distance_difference,distance_ratio
856340,2,2026-01-14 23:36:49,2026-01-15 00:01:19,1,9.0,2,N,138,170,1,...,Queens,-73.872833,40.774879,Murray Hill,Manhattan,-73.976944,40.747654,5.764178,3.235822,1.561367


In [50]:
def check_with_zone(row):
    pickup = str(row['pickup_zone'])
    dropoff = str(row['dropoff_zone'])
    fare_type = row['fare_type_id']

    if fare_type == 2:
        return 'JFK' in pickup.upper() or 'JFK' in dropoff.upper()

    if fare_type == 3:
        return 'NEWARK' in pickup.upper() or 'NEWARK' in dropoff.upper()

    if fare_type == 4:
        return (
            'NASSAU' in pickup.upper() or
            'NASSAU' in dropoff.upper() or
            'WESTCHESTER' in pickup.upper() or
            'WESTCHESTER' in dropoff.upper()
        )

    return pd.NA


In [53]:
df['fare_type_zone_mismatch'] = df.apply(check_with_zone, axis=1)

In [54]:
df['fare_type_zone_mismatch'].value_counts(dropna=False)

fare_type_zone_mismatch
<NA>     1759322
True       78737
False      15570
Name: count, dtype: int64

# check store_and_fwd_flag

In [56]:
df['store_and_fwd_flag'].value_counts(dropna=False)

store_and_fwd_flag
N    1852200
Y       1429
Name: count, dtype: int64

In [9]:
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].astype('boolean')

In [58]:
pd.crosstab(df['store_and_fwd_flag'], df['fare_type_zone_mismatch'], dropna=False)

fare_type_zone_mismatch,False,True,NaN
store_and_fwd_flag,,,
N,15558,78655,1757987
Y,12,82,1335


# save back to clean data file

In [10]:
df.to_parquet('../cleaned_data/clean_yellow_tripdata_2026-01.parquet', index=False)